# Assignment 20 - Movie Recommendation System

Content based recommender using TF-IDF and cosine similarity.

Dataset: TMDB 5000 Movie Dataset (`tmdb_5000_movies.csv`)
Kaggle link: https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata

## Part 1 - Data Preprocessing

### Task 1: Load and understand the dataset

In [1]:
import json
import re

import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("tmdb_5000_movies.csv")
print("Shape:", df.shape)

Shape: (4803, 20)


In [3]:
print(df.columns.tolist())

['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']


In [4]:
df.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   str    
 2   homepage              1712 non-null   str    
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   str    
 5   original_language     4803 non-null   str    
 6   original_title        4803 non-null   str    
 7   overview              4800 non-null   str    
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   str    
 10  production_countries  4803 non-null   str    
 11  release_date          4802 non-null   str    
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   str    
 15  status                4803 non-n

In [6]:
df.isnull().sum()

budget                     0
genres                     0
homepage                3091
id                         0
keywords                   0
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title                      0
vote_average               0
vote_count                 0
dtype: int64

In [7]:
print("duplicate titles:", df["title"].duplicated().sum())
df[["budget", "popularity", "runtime", "vote_average", "vote_count"]].describe()

duplicate titles: 3


,budget,popularity,runtime,vote_average,vote_count
count,4.803000e+03,4803.000000,4801.000000,4803.000000,4803.000000
mean,2.904504e+07,21.492301,106.875859,6.092172,690.217989
std,4.072239e+07,31.816650,22.611935,1.194612,1234.585891
min,0.000000e+00,0.000000,0.000000,0.000000,0.000000
25%,7.900000e+05,4.668070,94.000000,5.600000,54.000000
50%,1.500000e+07,12.921594,103.000000,6.200000,235.000000
75%,4.000000e+07,28.313505,118.000000,6.800000,737.000000
max,3.800000e+08,875.581305,338.000000,10.000000,13752.000000


In [8]:
# what the text columns look like for one movie
print(df.loc[0, "title"])
print(df.loc[0, "overview"])
print(df.loc[0, "genres"])
print(df.loc[0, "keywords"][:200])

Avatar
In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.
[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]
[{"id": 1463, "name": "culture clash"}, {"id": 2964, "name": "future"}, {"id": 3386, "name": "space war"}, {"id": 3388, "name": "space colony"}, {"id": 3679, "name": "society"}, {"id": 3801, "name": "


The item identifier is `title`.

For recommendations I am using these text columns:
- `overview` - the plot summary, this has most of the information
- `tagline` - one line about the movie
- `genres` - list of genres (stored as a JSON string)
- `keywords` - list of keywords (also JSON)

`homepage` has a lot of missing values but I don't need it. `overview` has 3 missing and `tagline` has around 800 missing, these are replaced with an empty string in the next step.

### Task 2: Text preprocessing

In [9]:
movies = df[["title", "overview", "genres", "keywords", "tagline"]].copy()

# a few titles are repeated, keeping the first one so that title lookup is unique
movies = movies.drop_duplicates(subset="title").reset_index(drop=True)

# missing values -> empty string
movies = movies.fillna("")
movies.isnull().sum()

title       0
overview    0
genres      0
keywords    0
tagline     0
dtype: int64

In [10]:
def get_names(text):
    # '[{"id": 28, "name": "Action"}, {"id": 878, "name": "Science Fiction"}]' -> 'Action ScienceFiction'
    # spaces inside a name are removed so "Science Fiction" is treated as one word
    try:
        items = json.loads(text)
    except (TypeError, ValueError):
        return ""
    return " ".join(i["name"].replace(" ", "") for i in items)

movies["genres"] = movies["genres"].apply(get_names)
movies["keywords"] = movies["keywords"].apply(get_names)
movies[["title", "genres", "keywords"]].head()

,title,genres,keywords
0,Avatar,Action Adventure Fantasy ScienceFiction,cultureclash future spacewar spacecolony socie...
1,Pirates of the Caribbean: At World's End,Adventure Fantasy Action,ocean drugabuse exoticisland eastindiatradingc...
2,Spectre,Action Adventure Crime,spy basedonnovel secretagent sequel mi6 britis...
3,The Dark Knight Rises,Action Crime Drama Thriller,dccomics crimefighter terrorist secretidentity...
4,John Carter,Action Adventure ScienceFiction,basedonnovel mars medallion spacetravel prince...


In [11]:
# combine everything into one text column
movies["tags"] = movies["overview"] + " " + movies["tagline"] + " " + movies["genres"] + " " + movies["keywords"]
movies["tags"][0]

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Enter the World of Pandora. Action Adventure Fantasy ScienceFiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d'

In [12]:
def clean(text):
    text = str(text).lower()                      # lowercase
    text = re.sub(r"[^a-z0-9\s]", " ", text)      # remove punctuation and special characters
    words = [w for w in text.split() if w not in ENGLISH_STOP_WORDS]   # remove stopwords
    return " ".join(words)

movies["clean_text"] = movies["tags"].apply(clean)
movies["clean_text"][0]

'22nd century paraplegic marine dispatched moon pandora unique mission torn following orders protecting alien civilization enter world pandora action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d'

In [13]:
movies[["title", "clean_text"]].head()

,title,clean_text
0,Avatar,22nd century paraplegic marine dispatched moon...
1,Pirates of the Caribbean: At World's End,captain barbossa long believed dead come life ...
2,Spectre,cryptic message bond s past sends trail uncove...
3,The Dark Knight Rises,following death district attorney harvey dent ...
4,John Carter,john carter war weary military captain s inexp...


Stopwords are taken from scikit-learn (`ENGLISH_STOP_WORDS`) since only pandas and scikit-learn are allowed, so no nltk.

## Part 2 - Text Vectorization

### Task 3: TF-IDF

In [14]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(movies["clean_text"])
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (4800, 5000)


In [15]:
print(tfidf.get_feature_names_out()[1000:1030])

['corporation' 'corpse' 'corrupt' 'corruption' 'cost' 'costume' 'couldn'
 'count' 'countries' 'country' 'countrymusic' 'countryside' 'county'
 'coup' 'couple' 'couples' 'courage' 'courageous' 'course' 'court'
 'courtcase' 'courtroom' 'cousin' 'cover' 'covert' 'cow' 'cowardliness'
 'cowboy' 'crab' 'crack']


- `max_features=5000` keeps the 5000 most frequent terms. Without a limit the vocabulary with bigrams goes into lakhs and most of those terms appear in only one movie.
- `ngram_range=(1, 2)` uses single words and pairs of words, so things like "new york" or "serial killer" are also captured.

### Task 4: Cosine similarity

In [16]:
similarity = cosine_similarity(tfidf_matrix)
print(similarity.shape)
similarity[:5, :5].round(3)

(4800, 4800)


array([[1.   , 0.053, 0.024, 0.019, 0.115],
       [0.053, 1.   , 0.02 , 0.003, 0.052],
       [0.024, 0.02 , 1.   , 0.009, 0.037],
       [0.019, 0.003, 0.009, 1.   , 0.012],
       [0.115, 0.052, 0.037, 0.012, 1.   ]])

**Why cosine similarity?**

Cosine similarity measures the angle between two vectors and not their length. Some movies have a long overview and some have only one line, so their TF-IDF vectors have very different magnitudes. With euclidean distance the long ones would look far from everything. With cosine only the direction matters, i.e. which words are important in both movies. The value is between 0 and 1 here (TF-IDF has no negative values), 1 means same content and 0 means nothing in common, so it is easy to sort on. It also works fast on sparse matrices.

## Part 3 - Recommendation Logic

### Task 5: Recommendation function

In [17]:
def recommend(item_name, top_n=5):
    if item_name not in movies["title"].values:
        print(item_name, "is not in the dataset")
        return []

    # index of the selected movie
    idx = movies[movies["title"] == item_name].index[0]

    # similarity of this movie with all the others
    scores = list(enumerate(similarity[idx]))

    # sort by score, first one is the movie itself so skip it
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n + 1]

    return [(movies["title"][i], round(float(s), 3)) for i, s in scores]

In [18]:
recommend("Avatar")

[('Lara Croft Tomb Raider: The Cradle of Life', 0.224), ('Titan A.E.', 0.181), ('Star Trek Into Darkness', 0.177), ('Battle: Los Angeles', 0.176), ('Falcon Rising', 0.175)]

In [19]:
recommend("The Dark Knight Rises")

[('The Dark Knight', 0.44), ('Batman Returns', 0.389), ('Batman Forever', 0.357), ('Batman Begins', 0.309), ('Batman', 0.297)]

In [20]:
recommend("Toy Story")

[('Toy Story 2', 0.47), ('Toy Story 3', 0.457), ('The 40 Year Old Virgin', 0.338), ('Man on the Moon', 0.265), ('Factory Girl', 0.227)]

In [21]:
recommend("Harry Potter and the Half-Blood Prince", top_n=8)

[('Harry Potter and the Order of the Phoenix', 0.29), ('Harry Potter and the Prisoner of Azkaban', 0.289), ('Harry Potter and the Goblet of Fire', 0.28), ('Harry Potter and the Chamber of Secrets', 0.237), ('Beautiful Creatures', 0.22), ("The Maid's Room", 0.216), ("Harry Potter and the Philosopher's Stone", 0.215), ('The Wicked Within', 0.202)]

In [22]:
recommend("Some random movie")

Some random movie is not in the dataset


[]

For Dark Knight Rises, Toy Story and Harry Potter the other movies from the same series come at the top, which is what I expected. Avatar is weaker, it gets general sci-fi / action movies with scores around 0.2 because there is no other movie in the data with a very similar overview. Toy Story also gets The 40 Year Old Virgin at number 3 because of the "toy" keywords, so the method only looks at common words and not the actual meaning.

The Streamlit app is in `app.py` and uses the same steps.